# rag-eval quickstart

This notebook walks through the whole rag-eval workflow on the **toy handbook demo**: an
original, fictional employee handbook for *Thimblewood Instruments* (30 short passages,
15 benchmark questions).

**install → validate → run both demo systems → report → compare → calibrate**

Steps that need an LLM (answering questions, judging groundedness, calibrating the judge)
are marked **needs key**. They run only when `RAG_EVAL_JUDGE_API_KEY` is set and are
skipped otherwise, so the notebook always runs top to bottom. Retrieval metrics (BM25 and
dense embeddings) need no LLM at all.

## 1. Install

From the repository root (once):

```bash
pip install -e ".[demo,dev]"   # demo = rank-bm25 + sentence-transformers
pip install jupyter            # only needed to open this notebook
```

The judge is any OpenAI-compatible endpoint, configured with environment variables, for
example a hosted provider:

```bash
export RAG_EVAL_JUDGE_BASE_URL=https://api.your-provider.example/v1
export RAG_EVAL_JUDGE_API_KEY=...        # unset = answer metrics are skipped below
export RAG_EVAL_JUDGE_MODEL=your-strong-model
```

The cell below finds the repository root and defines a small helper that runs the
`rag-eval` command line exactly as you would in a terminal.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import rag_eval

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
DEMO = "examples/toy_handbook"
HAS_KEY = bool(os.environ.get("RAG_EVAL_JUDGE_API_KEY"))


def cli(*args):
    """Run `rag-eval <args>` and print its output."""
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.run(
        [sys.executable, "-m", "rag_eval", "--quiet", *args],
        capture_output=True,
        text=True,
        encoding="utf-8",
        env=env,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"rag-eval {args[0]} failed")


print("rag-eval", rag_eval.__version__, "| judge key set:", HAS_KEY)

rag-eval 0.1.0 | judge key set: False


## 2. Validate the benchmark

`validate` checks every line of the questions file (unique IDs, gold chunk IDs for
answerable questions, none for unanswerable ones) and prints a summary. Freeze your
benchmark *before* you look at results: rag-eval records the file's SHA-256 in every run
and refuses to compare runs made on different files.

In [2]:
cli("validate", f"{DEMO}/questions.jsonl")

examples\toy_handbook\questions.jsonl: 15 valid question(s)
  answerable: 12   unanswerable: 3
  per qtype:
    lookup                   5
    multi_passage            4
    paraphrase               3
    unanswerable             3
OK



## 3. Run both demo systems (retrieval only, no LLM needed)

Each system is loaded through the plugin protocol from an import path to a factory:
`examples.toy_handbook.systems:make_bm25` (keyword search) and `...:make_dense`
(sentence-transformers embeddings). `--retrieval-only` scores Hit@k, Recall@k,
Precision@k and MRR without calling the generator or the judge. The dense system
downloads a small embedding model on first use.

In [3]:
for name in ("bm25", "dense"):
    cli(
        "run",
        "--system",
        f"examples.toy_handbook.systems:make_{name}",
        "--questions",
        f"{DEMO}/questions.jsonl",
        "--k",
        "3",
        "--out",
        f"runs/{name}_k3",
        "--retrieval-only",
    )


Run bm25_k3: 15 questions, 0 error(s), judge=none
  Hit@3                        0.917 [0.750, 1.000] (n=12)
  Recall@3                     0.833 [0.666, 1.000] (n=12)
  Precision@3                  0.361 [0.278, 0.444] (n=12)
  MRR                          0.819 [0.625, 1.000] (n=12)
  Latency p50 / p95 (ms)       0 / 0
Wrote runs\bm25_k3/config.json, results.jsonl, summary.json




Run dense_k3: 15 questions, 0 error(s), judge=none
  Hit@3                        1.000 [1.000, 1.000] (n=12)
  Recall@3                     0.917 [0.792, 1.000] (n=12)
  Precision@3                  0.389 [0.333, 0.472] (n=12)
  MRR                          0.903 [0.764, 1.000] (n=12)
  Latency p50 / p95 (ms)       21 / 30
Wrote runs\dense_k3/config.json, results.jsonl, summary.json



## 4. Full runs with answers (**needs key**)

Without `--retrieval-only`, each system also answers every question with an LLM, citing
`[chunk_id]`s, and the judge scores groundedness, citation precision, refusals and (with
`--correctness`) correctness against the human-written reference answers. Judge replies
are cached in `.rag_eval_cache.sqlite`, so re-running is free and reproducible.

In [4]:
FULL_RUNS = []
if HAS_KEY:
    for name in ("bm25", "dense"):
        out = f"runs/{name}_k3_full"
        cli(
            "run",
            "--system",
            f"examples.toy_handbook.systems:make_{name}",
            "--questions",
            f"{DEMO}/questions.jsonl",
            "--k",
            "3",
            "--out",
            out,
            "--correctness",
        )
        FULL_RUNS.append(out)
else:
    print("Skipped: set RAG_EVAL_JUDGE_API_KEY to run the answer step and the judge.")

Skipped: set RAG_EVAL_JUDGE_API_KEY to run the answer step and the judge.


## 5. Report

`report` writes a markdown file with every metric as `mean [95% CI] (n)`, per-qtype
tables, PNG charts and a failure log of the worst questions per run.

In [5]:
runs = ["runs/bm25_k3", "runs/dense_k3", *FULL_RUNS]
cli("report", *runs, "--out", "runs/REPORT.md")
text = Path("runs/REPORT.md").read_text(encoding="utf-8")
print(text.split("## Charts")[0])

Wrote runs\REPORT.md (2 run(s))

# rag-eval report

Generated by rag-eval 0.1.0. Every metric is shown as `mean [95% bootstrap CI] (n = questions it applies to)`.

| Run | System | k | Questions | Errors | Questions sha256 | Judge model | rag-eval | Created (UTC) |
|---|---|---|---|---|---|---|---|---|
| bm25_k3 | examples.toy_handbook.systems:make_bm25 | 3 | 15 | 0 | `2e74b203b7db` | none (retrieval only) | 0.1.0 | 2026-11-07T09:55:29+00:00 |
| dense_k3 | examples.toy_handbook.systems:make_dense | 3 | 15 | 0 | `2e74b203b7db` | none (retrieval only) | 0.1.0 | 2026-11-07T09:55:40+00:00 |

## Main results

| Run | Hit@3 | Recall@3 | MRR | Groundedness | Citation precision | Refusal accuracy | p50 latency (ms) | p95 latency (ms) |
|---|---|---|---|---|---|---|---|---|
| bm25_k3 | 0.917 [0.750, 1.000] (n=12) | 0.833 [0.666, 1.000] (n=12) | 0.819 [0.625, 1.000] (n=12) | - | - | - | 0 | 0 |
| dense_k3 | 1.000 [1.000, 1.000] (n=12) | 0.917 [0.792, 1.000] (n=12) | 0.903 [0.764, 1.000] (n=12) |

## 6. Compare two runs

`compare` pairs the runs question by question and bootstraps the difference B − A. With
only 12 answerable questions most confidence intervals include zero, and rag-eval says
so instead of declaring a winner.

In [6]:
cli("compare", "runs/bm25_k3", "runs/dense_k3")

# Comparison: B = dense_k3 vs A = bm25_k3

Differences are B - A in percentage points with a paired bootstrap 95% CI, computed over questions scored in both runs. A CI that includes zero means the difference is not clear at this sample size.

## Overall

- Hit@3 +8.3 pts [95% CI +0.0, +25.0] (n=12): the CI includes zero, so the difference is not clear at this sample size.
- Recall@3 +8.3 pts [95% CI +0.0, +25.0] (n=12): the CI includes zero, so the difference is not clear at this sample size.
- MRR +8.3 pts [95% CI -8.3, +29.2] (n=12): the CI includes zero, so the difference is not clear at this sample size.
- Precision@3 +2.8 pts [95% CI +0.0, +8.3] (n=12): the CI includes zero, so the difference is not clear at this sample size.

## qtype `lookup`

- Hit@3 +0.0 pts [95% CI +0.0, +0.0] (n=5): identical on every question.
- Recall@3 +0.0 pts [95% CI +0.0, +0.0] (n=5): identical on every question.
- MRR -10.0 pts [95% CI -30.0, +0.0] (n=5): the CI includes zero, so the difference is not

## 7. Calibrate the judge (**needs key**)

Groundedness is only as good as the judge. `calibrate` runs the judge's claim-verification
step on labelled claims and reports agreement and Cohen's kappa. Note: the demo's
`human_labels.jsonl` is a **Claude-labelled draft that still needs human review**; replace
or confirm the labels before trusting the result.

In [7]:
if HAS_KEY:
    cli("calibrate", "--labels", f"{DEMO}/human_labels.jsonl")
else:
    print("Skipped: set RAG_EVAL_JUDGE_API_KEY to calibrate the judge.")

Skipped: set RAG_EVAL_JUDGE_API_KEY to calibrate the judge.


## 8. The same from Python

Everything above is also available as a library. `run_live` takes a system object (or a
`"module:factory"` string) and returns the config, per-question results and summary.

In [8]:
from examples.toy_handbook.systems import make_bm25
from rag_eval import run_live

artifacts = run_live(make_bm25(), f"{DEMO}/questions.jsonl", k=3, retrieval_only=True)
print(json.dumps(artifacts.summary["overall"]["recall"], indent=2))
worst = sorted((r for r in artifacts.results if r.recall is not None), key=lambda r: r.recall)[:3]
for r in worst:
    print(r.question_id, r.qtype, "recall", r.recall, "retrieved", r.retrieved_ids)

{
  "mean": 0.8333333333333334,
  "low": 0.665625,
  "high": 1.0,
  "n": 12
}
q11 paraphrase recall 0.0 retrieved ['equipment-02', 'benefits-05', 'travel-01']
q08 multi_passage recall 0.5 retrieved ['security-02', 'equipment-01', 'security-04']
q09 multi_passage recall 0.5 retrieved ['equipment-02', 'equipment-04', 'benefits-04']
